In [1]:
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
import mlflow
import mlflow.sklearn
import time


In [2]:
df = pd.read_csv("../data/creditcard.csv")
print(f"Shape: {df.shape}")
print(f"Fraud rate: {df['Class'].mean():.4%}")

Shape: (284807, 31)
Fraud rate: 0.1727%


In [3]:
print(df.dtypes)

Time      float64
V1        float64
V2        float64
V3        float64
V4        float64
V5        float64
V6        float64
V7        float64
V8        float64
V9        float64
V10       float64
V11       float64
V12       float64
V13       float64
V14       float64
V15       float64
V16       float64
V17       float64
V18       float64
V19       float64
V20       float64
V21       float64
V22       float64
V23       float64
V24       float64
V25       float64
V26       float64
V27       float64
V28       float64
Amount    float64
Class       int64
dtype: object


In [4]:
scaler = StandardScaler()
df["Amount_scaled"] = scaler.fit_transform(df[["Amount"]])
df["Time_scaled"] = scaler.fit_transform(df[["Time"]])


In [5]:
features = [f"V{i}" for i in range(1, 29)] + ["Amount_scaled", "Time_scaled"]
X = df[features].values
y = df["Class"].values

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train frauds: {y_train.sum()} | Test frauds: {y_test.sum()}")

Train frauds: 394 | Test frauds: 98


In [7]:
mlflow.set_experiment("svm-kernel-comparison")

for kernel in ["linear", "rbf"]:
    print(f"\n{'='*50}")
    print(f"Training SVM — kernel: {kernel}")
    
    with mlflow.start_run(run_name=f"svm_{kernel}"):
        # NOTE: SVM is slow on 284K rows → use a subsample for speed
        # In real projects you'd use LinearSVC or SGDClassifier for full data
        idx = np.random.choice(len(X_train), size=10000, replace=False)
        X_sub = X_train[idx]
        y_sub = y_train[idx]

        t0 = time.time()
        clf = SVC(kernel=kernel, class_weight="balanced", probability=True, random_state=42)
        clf.fit(X_sub, y_sub)
        train_time = time.time() - t0

        y_pred  = clf.predict(X_test)
        y_proba = clf.predict_proba(X_test)[:, 1]
        auc     = roc_auc_score(y_test, y_proba)

        print(f"Train time : {train_time:.1f}s")
        print(f"AUC        : {auc:.4f}")
        print(classification_report(y_test, y_pred, target_names=["Normal", "Fraud"]))

        mlflow.log_params({"kernel": kernel, "subsample": 10000})
        mlflow.log_metrics({"auc": auc, "train_time_sec": train_time})
        mlflow.sklearn.log_model(clf, f"svm_{kernel}")


Training SVM — kernel: linear
Train time : 0.1s
AUC        : 0.9280
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     56864
       Fraud       0.78      0.74      0.76        98

    accuracy                           1.00     56962
   macro avg       0.89      0.87      0.88     56962
weighted avg       1.00      1.00      1.00     56962



2026/06/27 00:51:21 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



Training SVM — kernel: rbf
Train time : 0.5s
AUC        : 0.9641
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00     56864
       Fraud       0.70      0.41      0.52        98

    accuracy                           1.00     56962
   macro avg       0.85      0.70      0.76     56962
weighted avg       1.00      1.00      1.00     56962



2026/06/27 00:51:26 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
